In [2]:
import os, sys, math
import numpy as np
from dataclasses import dataclass
from tdmclient import ClientAsync

# ---------------------------------------------------------
# Make sure we can import your src/ folder with calibration
# ---------------------------------------------------------
sys.path.insert(0, os.path.join(os.getcwd(), "src"))

try:
    from local_occupancy import (
        sensor_measurements,
        sensor_distances,
        sensor_pos_from_center,
        sensor_angles,
    )
except ImportError:
    # Not used in this heuristic version, but keep dummy values to avoid crashes.
    sensor_measurements = np.array([0, 4000])
    sensor_distances = np.array([100.0, 2.0])
    sensor_pos_from_center = [(0.0, 0.0)] * 7
    sensor_angles = [0.0] * 7


# -------------------------------
# Helpers
# -------------------------------

def wrap_to_pi(angle: float) -> float:
    """Wrap any angle (rad) to (-pi, pi]."""
    return (angle + math.pi) % (2.0 * math.pi) - math.pi


def vec_to_polar_robot(x_left, y_forward):
    """
    Convert a robot-frame vector [x_left, y_forward] into (mag, angle_deg).

    angle_deg:
        0°  -> straight ahead
        >0° -> steer left
        <0° -> steer right
    """
    mag = math.hypot(x_left, y_forward)
    ang_deg = math.degrees(math.atan2(x_left, y_forward))
    return mag, ang_deg


# -------------------------------
# Local navigation: path lookahead
# -------------------------------

@dataclass
class LocalNavConfig:
    lookahead_dist_m: float = 0.15
    max_lookahead_points: int = 30


class LocalNavigator:
    """
    Only responsible for:
      - frame transforms (world <-> robot)
      - finding lookahead point on the path
    """

    def __init__(self, cfg: LocalNavConfig):
        self.cfg = cfg
        self.path_idx = 0

    @staticmethod
    def world_to_robot(pose, point_world):
        """
        WORLD: x to the right, y downward (your convention),
        ROBOT: x_left, y_forward.
        """
        x_r, y_r, theta = pose
        px, py = point_world
        dx = px - x_r
        dy = py - y_r
        c = math.cos(theta)
        s = math.sin(theta)
        # y_forward along robot heading
        y_forward = c * dx + s * dy
        # x_left  = 90° CCW from heading
        x_left = -s * dx + c * dy
        return np.array([x_left, y_forward], dtype=float)

    @staticmethod
    def robot_to_world(pose, point_robot):
        x_r, y_r, theta = pose
        x_left, y_forward = point_robot
        c = math.cos(theta)
        s = math.sin(theta)
        dx_w = -s * x_left + c * y_forward
        dy_w =  c * x_left + s * y_forward
        return np.array([x_r + dx_w, y_r + dy_w], dtype=float)

    def find_lookahead_point(self, pose, path):
        """
        Pure-pursuit style:
        Find closest path index, then move forward along path
        until lookahead_dist_m.
        """
        x, y, theta = pose
        if path is None or path.shape[0] == 0:
            return None, None

        start_idx = self.path_idx
        end_idx = min(path.shape[0], start_idx + self.cfg.max_lookahead_points)
        segment = path[start_idx:end_idx]
        dists2 = (segment[:, 0] - x) ** 2 + (segment[:, 1] - y) ** 2
        k_local = int(np.argmin(dists2))
        k_global = start_idx + k_local
        self.path_idx = k_global

        target_idx = k_global
        accum = 0.0
        while (
            target_idx + 1 < path.shape[0]
            and accum < self.cfg.lookahead_dist_m
            and target_idx - k_global < self.cfg.max_lookahead_points
        ):
            p0 = path[target_idx]
            p1 = path[target_idx + 1]
            step = float(np.linalg.norm(p1 - p0))
            accum += step
            target_idx += 1

        LA_point = path[target_idx]
        return LA_point, k_global


# -------------------------------
# Go-to-goal controller
# -------------------------------

@dataclass
class GoToGoalGains:
    Kv: float = 2.0
    Komega: float = 3.0
    v_max: float = 0.3
    w_max: float = 2.0


@dataclass
class ThymioKinematics:
    wheel_radius: float = 0.022
    half_axle: float = 0.0475
    motor_gain: float = 10.0
    motor_max: int = 300


class GoToGoalController:
    def __init__(self, gains: GoToGoalGains, kin: ThymioKinematics):
        self.g = gains
        self.kin = kin

    def compute_unicycle_cmd(self, pose, goal_xy):
        """
        Classic go-to-goal to a point in WORLD frame.
        """
        x, y, theta = pose
        xg, yg = goal_xy
        dx = xg - x
        dy = yg - y

        # Distance in WORLD
        rho = math.hypot(dx, dy)

        # Express goal in ROBOT frame
        y_forward = math.cos(theta) * dx + math.sin(theta) * dy
        x_left    = -math.sin(theta) * dx + math.cos(theta) * dy

        # Heading error
        alpha = math.atan2(x_left, y_forward)
        alpha = wrap_to_pi(alpha)

        # Small deadzone on angle
        alpha_dead = 5.0 * math.pi / 180.0
        if abs(alpha) < alpha_dead:
            alpha = 0.0

        v = self.g.Kv * rho
        w = self.g.Komega * alpha

        v = max(-self.g.v_max, min(self.g.v_max, v))
        w = max(-self.g.w_max, min(self.g.w_max, w))

        return v, w, rho, alpha

    def unicycle_to_wheels(self, v, w):
        r = self.kin.wheel_radius
        L = self.kin.half_axle

        phi_L = (v / r) - (L / r) * w
        phi_R = (v / r) + (L / r) * w

        uL = int(self.kin.motor_gain * phi_L)
        uR = int(self.kin.motor_gain * phi_R)

        uL = max(-self.kin.motor_max, min(self.kin.motor_max, uL))
        uR = max(-self.kin.motor_max, min(self.kin.motor_max, uR))

        return uL, uR


# -------------------------------
# Heuristic obstacle avoidance (your original logic)
# -------------------------------

def apply_side_heuristic(sensor_vals, w, min_turn=1.5, side_thr=1200):
    """
    Enforce the sign and MINIMUM magnitude of w based on which outer
    front sensors see an obstacle.

    Thymio prox.horizontal indices:
        0: front left
        1: front left-center
        2: front center
        3: front right-center
        4: front right
        5: rear left
        6: rear right

    - If right side is stronger -> force a LEFT turn (w > 0)
    - If left side is stronger  -> force a RIGHT turn (w < 0)
    """
    s = sensor_vals

    left_side = max(s[0], s[1])
    right_side = max(s[3], s[4])

    # nothing significant
    if left_side < side_thr and right_side < side_thr:
        return w

    if right_side > left_side and right_side >= side_thr:
        # obstacle on RIGHT -> force LEFT turn (w > 0)
        if w <= 0:
            w = min_turn
        else:
            w = max(w, min_turn)
    elif left_side > right_side and left_side >= side_thr:
        # obstacle on LEFT -> force RIGHT turn (w < 0)
        if w >= 0:
            w = -min_turn
        else:
            w = min(w, -min_turn)

    return w


def apply_center_heuristic(sensor_vals, v, w, min_turn=1.8, center_thr=1500):
    """
    Special handling when the FRONT-CENTER sensor (index 2) sees something.

    - Strong turn left/right depending on which side also sees the obstacle more.
    - Slightly reduce v so that the robot turns more aggressively.
    """
    s = sensor_vals
    center_val = s[2]

    if center_val < center_thr:
        return v, w

    left_side = max(s[0], s[1])
    right_side = max(s[3], s[4])

    if left_side < center_thr and right_side < center_thr:
        # keep sign of w, enforce minimum magnitude
        if w >= 0:
            w = max(w, min_turn)
        else:
            w = min(w, -min_turn)
    else:
        if right_side >= left_side:
            # more on the right -> turn LEFT (w>0)
            if w <= 0:
                w = min_turn
            else:
                w = max(w, min_turn)
        else:
            # more on the left -> turn RIGHT (w<0)
            if w >= 0:
                w = -min_turn
            else:
                w = min(w, -min_turn)

    # slow down forward speed while dodging
    v = min(v, 0.05)

    return v, w


# -------------------------------
# Build straight path
# -------------------------------

# Straight path along +x in WORLD
path_length = 1.0   # 1 m
num_points = 60
x_path = np.linspace(0.0, path_length, num_points)
y_path = np.zeros(num_points)
path = np.vstack((x_path, y_path)).T


# -------------------------------
# Thymio connection
# -------------------------------

client = ClientAsync()
node = await client.wait_for_node()
await node.lock()
await node.wait_for_variables({"prox.horizontal"})

def motors(l_speed=0, r_speed=0):
    return {
        "motor.left.target": [int(l_speed)],
        "motor.right.target": [int(r_speed)],
    }


# -------------------------------
# Main loop: path following + heuristic avoidance
# -------------------------------

async def follow_segment_with_heuristic_avoidance(
    path,
    dt=0.10,
    max_time_s=25.0,
    goal_tol_m=0.02,
):
    """
    Follow a straight path with local obstacle avoidance using ONLY heuristics.

    - Base command: go-to-goal toward lookahead point on the path.
    - Avoidance: apply_side_heuristic + apply_center_heuristic
      to modify the angular speed (and sometimes v).
    """
    cfg_nav = LocalNavConfig(
        lookahead_dist_m=0.15,
        max_lookahead_points=30,
    )
    navigator = LocalNavigator(cfg_nav)

    gains = GoToGoalGains(
        Kv=2.0,
        Komega=3.0,
        v_max=0.30,
        w_max=2.0,
    )
    kin = ThymioKinematics()
    g2g = GoToGoalController(gains, kin)

    # initial pose at start of path, pointing along +x
    pose = np.array([path[0, 0], path[0, 1], 0.0], dtype=float)
    final_goal = path[-1]
    n_steps = int(max_time_s / dt)

    print("Starting path following with heuristic obstacle avoidance (NO F_rep)...")

    for k in range(n_steps):
        # 1) Read sensors
        sensor_vals = list(node.v.prox.horizontal)

        # 2) Compute lookahead point on path
        LA_world, _ = navigator.find_lookahead_point(pose, path)
        if LA_world is None:
            print("No lookahead point (empty path?), stopping.")
            break

        # For debug: vector to LA in robot frame (= 'bring me back' direction)
        F_att_robot = navigator.world_to_robot(pose, LA_world)
        mag_att, ang_att_deg = vec_to_polar_robot(F_att_robot[0], F_att_robot[1])

        # 3) Base go-to-goal command toward lookahead
        v_nom, w_nom, rho, alpha = g2g.compute_unicycle_cmd(pose, LA_world)

        # 4) Apply heuristics to modify w (and sometimes v)
        v_cmd, w_cmd = v_nom, w_nom

        # side sensors choose turning side & min |w|
        w_cmd = apply_side_heuristic(sensor_vals, w_cmd,
                                     min_turn=1.5, side_thr=1200)

        # center sensor -> strong avoidance
        v_cmd, w_cmd = apply_center_heuristic(sensor_vals, v_cmd, w_cmd,
                                              min_turn=1.8, center_thr=1500)

        # Optional: if something is really close, reduce v a bit globally
        if max(sensor_vals[:5]) > 2000:
            v_cmd = min(v_cmd, 0.08)

        # 5) Map (v_cmd, w_cmd) to wheel commands
        uL, uR = g2g.unicycle_to_wheels(v_cmd, w_cmd)
        await node.set_variables(motors(uL, uR))

        # 6) Integrate pose with (v_cmd, w_cmd)
        pose[0] += v_cmd * math.cos(pose[2]) * dt
        pose[1] += v_cmd * math.sin(pose[2]) * dt
        pose[2] = wrap_to_pi(pose[2] + w_cmd * dt)

        # Distance to final goal
        dx = final_goal[0] - pose[0]
        dy = final_goal[1] - pose[1]
        dist_to_goal = math.hypot(dx, dy)

        # Debug every few steps
        if k % 5 == 0:
            print(
                f"step {k} | dist_to_goal={dist_to_goal:.3f} m | "
                f"sensors={sensor_vals}"
            )
            print(
                f"  F_att (to path) mag={mag_att:.2f}, ang={ang_att_deg:+.1f}° | "
                f"v_nom={v_nom:.3f}, w_nom={w_nom:.3f} | "
                f"v_cmd={v_cmd:.3f}, w_cmd={w_cmd:.3f} | "
                f"uL={uL}, uR={uR}"
            )
            print()

        if dist_to_goal < goal_tol_m:
            print(f"Reached goal within {goal_tol_m*100:.1f} cm at step {k}")
            break

        await client.sleep(dt)

    # stop at the end
    await node.set_variables(motors(0, 0))
    print("Stopped.")





In [4]:
await follow_segment_with_heuristic_avoidance(path)

Starting path following with heuristic obstacle avoidance (NO F_rep)...
step 0 | dist_to_goal=0.992 m | sensors=[2611, 2183, 0, 0, 0, 0, 0]
  F_att (to path) mag=0.15, ang=+0.0° | v_nom=0.300, w_nom=0.000 | v_cmd=0.080, w_cmd=-1.500 | uL=68, uR=3

step 5 | dist_to_goal=0.894 m | sensors=[0, 0, 1993, 0, 0, 0, 0]
  F_att (to path) mag=0.15, ang=-5.0° | v_nom=0.300, w_nom=0.000 | v_cmd=0.050, w_cmd=1.800 | uL=-16, uR=61

step 10 | dist_to_goal=0.869 m | sensors=[0, 0, 2265, 0, 0, 0, 0]
  F_att (to path) mag=0.15, ang=+4.1° | v_nom=0.291, w_nom=0.000 | v_cmd=0.050, w_cmd=1.800 | uL=-16, uR=61

step 15 | dist_to_goal=0.844 m | sensors=[0, 0, 2566, 0, 0, 0, 0]
  F_att (to path) mag=0.15, ang=-6.7° | v_nom=0.300, w_nom=-0.353 | v_cmd=0.050, w_cmd=-1.800 | uL=61, uR=-16

step 20 | dist_to_goal=0.819 m | sensors=[0, 0, 2942, 0, 0, 0, 0]
  F_att (to path) mag=0.15, ang=+2.9° | v_nom=0.293, w_nom=0.000 | v_cmd=0.050, w_cmd=1.800 | uL=-16, uR=61

step 25 | dist_to_goal=0.795 m | sensors=[0, 0, 330

In [3]:
await node.set_variables(motors(0, 0))
